**Note:** This notebook is designed for **Google Colab**.

If you see the Colab logo <span style='vertical-align:bottom;'><img src='https://colab.research.google.com/img/colab_favicon_256px.png' width='40' alt='Colab logo'></span> in the top-left corner, you're all set! Please **proceed to Section 1**.

If you don't see the logo (e.g., you are on GitHub), please click the button below to open it in the correct environment:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mparrott-at-wiris/aimodelshare/blob/master/notebooks/Etica_en_Joc_Justice_Challenge.ipynb)

# **Advanced Justice & Equity Challenge: Build & Submit Custom Models**

Welcome to the **Advanced Pathway** of the Ethics at Play (Ètica en Joc) Justice Challenge. 

**Who is this for?** 
This notebook is designed for participants with Python experience (e.g., Scikit-Learn, TensorFlow, PyTorch). Instead of using the gamified apps, you will build, train, and submit your own machine learning models directly to the competition leaderboard.

**The Goal:** 
Train a model to predict recidivism risk (the likelihood of re-offending) using the COMPAS dataset, while balancing accuracy and fairness.

## 🚀 **Quick Start Guide**

To participate in the challenge, complete these 5 steps:

1.  **Install Libraries:** Run the setup cell to install `aimodelshare`.
2.  **Get the Data:** Run the data loading cell to retrieve the COMPAS dataset.
3.  **Train Your Model:** Use the provided Scikit-Learn Pipeline example or write your own custom training code.
4.  **Connect:** Link this notebook to the Justice Challenge Leaderboard.
5.  **Submit:** Send your predictions to the leaderboard to see your score.

**Ready? Click the ▶ Play Button on the first cell below to get started.**

---
# **Step 1: Installation**

We need to install the `aimodelshare` library to connect to the competition backend.

In [ ]:
# Install the aimodelshare library
print("Installing required libraries...")
!pip install aimodelshare --upgrade -q --no-warn-script-location > /dev/null 2>&1
print("✅ Installation complete!")

---
# **Step 2: Load Data**

We will use the **COMPAS** dataset, which is the standard dataset used for this challenge. 

In this step, we will load the raw data, drop missing values, and split it into training and testing sets. We will handle the feature engineering (converting text to numbers) in the next step using a Pipeline.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Load the dataset
url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
data = pd.read_csv(url)

# 2. Select features relevant to the challenge
features = ['sex', 'age', 'race', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count', 'c_charge_degree']
target = 'two_year_recid'

# 3. Basic Cleaning (Drop missing values)
# We drop rows with missing values before splitting to ensure data quality
df = data[features + [target]].dropna()

# 4. Split into X (Features) and y (Target)
X = df.drop(target, axis=1)
y = df[target]

# 5. Split into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("✅ Data loaded and split!")
print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")
print("\nFirst 5 rows of training data:")
X_train.head()

---
# **Step 3: Train Model with Pipeline**

We will use a **Scikit-Learn Pipeline** to streamline preprocessing and modeling. 

This pipeline will:
1.  **One-Hot Encode** categorical columns (Race, Sex, Charge Degree).
2.  **Scale** numerical columns (Age, Priors, etc.).
3.  **Train** a Logistic Regression classifier.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Define feature groups
# Numerical features will be scaled
numeric_features = ['age', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count']

# Categorical features will be One-Hot Encoded
categorical_features = ['sex', 'race', 'c_charge_degree']

# 2. Create Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# 3. Create Pipeline (Preprocessor + Model)
# You can replace LogisticRegression with any other sklearn model (e.g., RandomForestClassifier)
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

# 4. Train the pipeline
pipeline.fit(X_train, y_train)

# 5. Generate predictions on the test set
predictions = pipeline.predict(X_test)

# 6. Evaluate accuracy
accuracy = accuracy_score(y_test, predictions)
print(f"✅ Model Trained! Accuracy: {accuracy:.2%}")

---
# **Step 4: Connect to the Leaderboard**

This step connects your notebook to the specific backend for the Justice & Equity Challenge. 

*Note: You will be prompted to enter a username and password. If you don't have one, check with your instructor or the challenge website.*

In [ ]:
from aimodelshare.aws import set_credentials
from aimodelshare.playground import Competition

# The specific Model Playground URL for the Justice Challenge
my_playground_url = "https://cf3wdpkg0d.execute-api.us-east-1.amazonaws.com/prod/m"

# Set your credentials (pop-up will appear)
set_credentials(apiurl=my_playground_url)

# Connect to the competition
playground = Competition(my_playground_url)

---
# **Step 5: Submit & Check Results**

Submit your predictions to the leaderboard.

In [ ]:
# 1. Submit your predictions
# Note: We pass None for model and preprocessor because we are only submitting predictions for evaluation
playground.submit_model(
    model=None,
    preprocessor=None,
    prediction_submission=predictions,
    input_dict={
        "description": "Logistic Regression with Sklearn Pipeline", 
        "tags": "sklearn, logistic_regression, advanced_pathway, pipeline"
    }
)

print("✅ Predictions submitted successfully!")

# 2. Check the leaderboard
print("Loading leaderboard...")
leaderboard = playground.get_leaderboard()
playground.stylize_leaderboard(leaderboard)